# 00 — Project Data Setup

Run this notebook whenever you start the project on a new Colab runtime. It downloads GTZAN only if it is not already present and loads the committed leakage-safe split manifest.


In [ ]:
# 1. Clone the latest project branch
!rm -rf /content/music-genre-classification
!git clone -b feature/deep-learning https://github.com/SaimaKapoor/music-genre-classification.git /content/music-genre-classification

import sys
sys.path.insert(0, "/content/music-genre-classification")
print("Repository ready")


In [ ]:
# 2. Install the runtime dependencies
!pip -q install librosa soundfile pandas scikit-learn matplotlib
print("Dependencies ready")


In [ ]:
# 3. Download GTZAN automatically if this runtime does not have it
from pathlib import Path
import sys

sys.path.insert(0, "/content/music-genre-classification")
from src.common.gtzan_setup import ensure_gtzan

GTZAN_ROOT = ensure_gtzan("/content/gtzan")


In [ ]:
# 4. Locate the committed split manifest
from pathlib import Path
import pandas as pd

MANIFEST = Path("/content/music-genre-classification/data/split_manifest.csv")

if not MANIFEST.exists():
    raise FileNotFoundError(
        "data/split_manifest.csv is missing from the GitHub branch. "
        "Commit the manifest to the repository before running the project."
    )

manifest = pd.read_csv(MANIFEST)
print("Manifest:", MANIFEST)
print("Rows:", len(manifest))
print(manifest["split"].value_counts())
print("\nGenres:")
print(manifest["genre"].value_counts().sort_index())


In [ ]:
# 5. Sanity-check that the manifest paths resolve to real audio files
from pathlib import Path

paths = manifest["relative_path"].astype(str).tolist()
roots = [Path("/content/gtzan"), Path("/content/gtzan/genres")]

best_root = max(
    roots,
    key=lambda r: sum((r / p).is_file() for p in paths)
)

matched = sum((best_root / p).is_file() for p in paths)
print("Audio root:", best_root)
print(f"Manifest audio paths found: {matched}/{len(paths)}")

if matched != len(paths):
    missing = [p for p in paths if not (best_root / p).is_file()][:10]
    raise FileNotFoundError(f"Some manifest audio paths are missing. Examples: {missing}")

print("DATA SETUP COMPLETE!")
